In [1]:
# Lets create a table with name 'images' in lanceDB. We shall add embedding for each image here.
import lancedb
from lancedb.pydantic import LanceModel, Vector
import numpy as np


class mySchema(LanceModel):
    id: str  
    vector: Vector(512)  # type: ignore 

def create_table(db_path, table_name= "images"):
    db = lancedb.connect(db_path)

    if table_name not in db.table_names():
        tbl = db.create_table(
            table_name,
            schema=mySchema,
            data=[
                {"id": "rand_0", "vector": np.random.rand(512).astype(np.float32)}
            ],
        )

    else:
        tbl = db.open_table(table_name)
        schema_fields = [field.name for field in tbl.schema]
        if schema_fields != list(mySchema.model_fields.keys()):
            raise RuntimeError(f"Table {table_name} has a different schema.")
        
    table = tbl
    return table

In [2]:
def dummy_run(config, bindings, output_buffer, timeout_ms = 1000,):
    config.run([bindings], timeout_ms)
    vec = bindings.output().get_buffer()
    return vec


In [3]:
from PIL import Image
# set input output buffer, run model and return embedding vec
def process_image(base_folder, rel_path, config, bindings, output_buffer, timeout_ms = 1000):

    # Resize image and 
    image_path = base_folder / rel_path
    image = Image.open(image_path).convert("RGB").resize((224, 224))
    input_buffer = np.asarray(image).astype(np.uint8)  # uint8, no batch dimension

    # IO Binding
    bindings.input().set_buffer(input_buffer)
    bindings.output().set_buffer(output_buffer)
    
    # Run model synchronous inference and access the output buffers
    
    return dummy_run(config, bindings, output_buffer)

In [4]:
from pathlib import Path
import time

profile_batch_size = 100
max_images = 5000

def process_image_dir(base_folder:Path, relative_path, config, bindings, output_buffer, vec_table):
    path:Path = base_folder / relative_path
   
    if path.is_dir():
        #print(f"processing all the files in {path}")
        # Recursively iterate to all files and call process_image
        start = time.perf_counter()
        batch_start = start
        for ext in ("*.jpg",):  # , "*.jpeg", "*.png"
            all_files = list(path.rglob(ext))
            files_2_process = all_files[:max_images]
            total = len(files_2_process)
            for i, image_path in enumerate(files_2_process):
                #print(f"processing {i+1}/{total}")
                rel_path = (image_path.resolve()).relative_to(base_folder)
                vec = process_image(base_folder, rel_path, config, bindings, output_buffer)
                vec_table.add([{ "id": str(rel_path), "vector": vec }])
                if i %profile_batch_size == 0:
                    end = time.perf_counter()
                    print(f"Images processed: {i+1: 6} / {total: 6} | Time taken: {end - start:.4f} seconds | batch time {end - batch_start:.4f} seconds")
                    batch_start = time.perf_counter()
                    
    else:
        print(f"processing {relative_path}")
        vec = process_image(base_folder, relative_path, config, bindings, output_buffer)
        vec_table.add([{ "id": str(relative_path), "vector": vec }])

In [5]:
from hailo_platform import VDevice, HailoSchedulingAlgorithm
# Create a inference mode, configure, create binding and then invoke process_image_dir
def process(hef, base_folder, relative_path, vec_table):
    params = VDevice.create_params()
    params.scheduling_algorithm = HailoSchedulingAlgorithm.ROUND_ROBIN
    with VDevice(params) as vdevice:
        # Create an infer model from an HEF:
        infer_model = vdevice.create_infer_model(hef)

        # Configure the infer model and create bindings for it
        with infer_model.configure() as config:
            bindings = config.create_bindings()
            output_buffer = np.empty(list(infer_model.output().shape), dtype=np.uint8)
            process_image_dir(base_folder, relative_path,  config, bindings, output_buffer, vec_table)

In [6]:
import time

DB_PATH = "image_embeddings2.lance"
# Setting up paths.
image_folder = Path("/home/anandas/test_images/Datewise/")
base_folder = Path("/home/anandas/test_images")
MODEL_HEF = "resnet_v1_18_feature.hef"

start = time.perf_counter()
table = create_table(DB_PATH)
process(hef=MODEL_HEF, base_folder=base_folder, relative_path=Path('Datewise'), vec_table=table)
end = time.perf_counter()

print(f"Time taken: {end - start:.4f} seconds")


Images processed:      1 /   5000 | Time taken: 1.5772 seconds | batch time 1.5772 seconds
Images processed:    101 /   5000 | Time taken: 113.1660 seconds | batch time 111.5885 seconds


KeyboardInterrupt: 